## Импорт библиотек

In [ ]:
from pathlib import Path
import os
from diffusers import StableDiffusionPipeline
from tqdm import tqdm
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    efficientnet_b0, EfficientNet_B0_Weights,
    convnext_tiny, ConvNeXt_Tiny_Weights,
    resnet50, ResNet50_Weights
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from PIL import Image
import cv2

In [ ]:
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
DEVICE = torch.device("mps")
IMAGE_SIZE = 224

In [ ]:
MODEL_DIR = Path("./weather/models")
OWN_DATA_DIR = Path("./own_data")
GENERATED_DIR = Path("./generated_data")

In [ ]:
MODELS = ["pretrained_convnext_tiny", "pretrained_efficientnet_b0", "pretrained_resnet50"]
CLASSES = ["dew", "fogsmog", "frost", "glaze", "hail", "lightning", "rain", "rainbow", "rime", "sandstorm", "snow"]

# Валидация

Функция построения модели

In [ ]:
def build_pretrained_model(model_name: str, num_classes: int = 11):
    if "efficientnet_b0" in model_name:
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(nn.Dropout(0.35), nn.Linear(in_features, num_classes))
    
    elif "convnext_tiny" in model_name:
        model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)
    
    elif "resnet50" in model_name:
        model = resnet50(weights=ResNet50_Weights.DEFAULT)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(0.35), nn.Linear(in_features, num_classes))
    else:
        raise ValueError(f"Неизвестная модель: {model_name}")
    
    return model

Загрузка обученных моделей

In [ ]:
loaded_models = {}
for name in MODELS:
    model_dir = MODEL_DIR / name
    state_dict_path = model_dir / "best_model.pt"
    
    model = build_pretrained_model(name, num_classes=11)
    model.load_state_dict(torch.load(state_dict_path, map_location=DEVICE, weights_only=True))
    model = model.to(DEVICE)
    model.eval()
    loaded_models[name] = model
    print(f"Загружена: {name} из {state_dict_path}")

## Свои изображения

Датасет для своих изображений

In [ ]:
class OwnImagesDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        self.classes = sorted([d.name for d in self.root_dir.iterdir() if d.is_dir()])
        
        for class_dir in self.root_dir.iterdir():
            if class_dir.is_dir():
                for img_path in class_dir.iterdir():
                    if img_path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}:
                        self.samples.append((img_path, class_dir.name))
        
        print(f"Найдено {len(self.samples)} изображений в {len(self.classes)} классах")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, class_name = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = self.classes.index(class_name)
        return image, label, str(img_path)

In [ ]:
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.224])
])

In [ ]:
own_dataset = OwnImagesDataset(OWN_DATA_DIR, transform=eval_transform)
own_loader = DataLoader(own_dataset, batch_size=32, shuffle=False)

Функция оценки модели

In [ ]:
def evaluate_on_own(model, loader, device, dataset_classes=None):
    model.eval()
    y_true, y_pred, all_probs, all_paths = [], [], [], []
    
    with torch.no_grad():
        for images, labels, paths in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            preds = torch.argmax(logits, dim=1).cpu().tolist()
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            
            y_true.extend(labels.cpu().tolist())
            y_pred.extend(preds)
            all_probs.extend(probs)
            all_paths.extend(paths)
    
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    print(f"Accuracy: {acc:.4f}, F1-macro: {f1:.4f}")
    
    target_names = dataset_classes if dataset_classes else [str(i) for i in range(len(set(y_true)))]
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))
    
    return {"y_true": y_true, "y_pred": y_pred, "probs": all_probs, "paths": all_paths, "acc": acc, "f1": f1}

Оценка на своих данных

In [ ]:
results = {}
for name, model in loaded_models.items():
    print(f"\n{'='*50}")
    print(f"Модель: {name}")
    print('='*50)
    results[name] = evaluate_on_own(model, own_loader, DEVICE, dataset_classes=CLASSES)

Лучшая модель — ResNet50 (Accuracy 0.88, F1-macro 0.8801): показывает наиболее сбалансированные результаты по всем классам. Классы rain, frost, rime определяются хуже всего.

## Аугментированные данные

Оценка на аугментированных (вручную искажённых) данных

In [ ]:
manual_augment_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 20, IMAGE_SIZE + 20)),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.5, 0.7)),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1),
    transforms.RandomPerspective(distortion_scale=0.4, p=1.0),
    transforms.GaussianBlur(kernel_size=7, sigma=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.224]),
])

In [ ]:
aug_dataset = OwnImagesDataset(OWN_DATA_DIR, transform=manual_augment_transform)
aug_loader = DataLoader(aug_dataset, batch_size=32, shuffle=False)

In [ ]:
print("\n" + "="*60)
print("Оценка на аугментированных (вручную искажённых) данных")
print("="*60)

aug_results = {}
for name, model in loaded_models.items():
    print(f"\n{'='*50}")
    print(f"Модель: {name}")
    print('='*50)
    aug_results[name] = evaluate_on_own(model, aug_loader, DEVICE, dataset_classes=CLASSES)

Падение качества значительное, что говорит о низкой устойчивости предобученных моделей к искажениям

## Генерация данных Stable Diffusion

In [ ]:
GENERATED_DIR.mkdir(exist_ok=True)

In [ ]:
PROMPTS = {
    "dew": "close-up photo of morning dew drops on green grass leaves, water droplets, nature, realistic, high quality",
    "fogsmog": "thick fog covering the city street, low visibility, hazy atmosphere, smog, realistic photo",
    "frost": "frost patterns on a window glass in winter, ice crystals, frozen fern patterns, close-up, realistic",
    "glaze": "ice glaze coating tree branches and wires, frozen rain, glaze ice, winter weather, realistic photo",
    "hail": "large hailstones on the ground after a storm, ice balls, severe weather, realistic photo",
    "lightning": "dramatic lightning bolt striking across dark storm clouds, night sky, realistic weather photography",
    "rain": "heavy rain falling on city street, raindrops, wet road reflections, overcast sky, realistic photo",
    "rainbow": "bright colorful rainbow arching over green field after rain, vivid colors, nature, realistic photo",
    "rime": "white rime ice covering tree branches, feathery frost, frozen fog, winter wonderland, realistic photo",
    "snow": "heavy snowfall in a forest, snow-covered trees, winter scene, snowflakes, realistic photo"
}

Загружаем модель

In [ ]:
print("Загрузка Stable Diffusion модели...")
pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float32,
    use_safetensors=True
)
pipe = pipe.to(DEVICE)
pipe.set_progress_bar_config(disable=False)

In [ ]:
num_images_per_class = 20
generator = torch.Generator(device=DEVICE).manual_seed(SEED)

In [ ]:
print(f"\nГенерация изображений для {len(CLASSES)} классов...")
for class_name in CLASSES:
    gen_class_dir = GENERATED_DIR / class_name
    gen_class_dir.mkdir(exist_ok=True)
    prompt = PROMPTS.get(class_name, f"photo of {class_name} weather, realistic")
    print(f"\n[{class_name}] prompt: {prompt[:80]}...")
    
    # Создаем стабильный seed на основе class_name
    class_seed = abs(hash(class_name)) % 10000  # abs() для положительного числа
    
    for i in tqdm(range(num_images_per_class), desc=f"  {class_name}"):
        # Создаем генератор отдельно от вызова pipe
        seed = SEED + i + class_seed
        generator = torch.Generator(device=DEVICE).manual_seed(seed)
        
        image = pipe(
            prompt=prompt,
            num_inference_steps=30,
            guidance_scale=7.5,
            generator=generator,
        ).images[0]
        
        image.save(gen_class_dir / f"sd_gen_{i}.png")

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
axes = axes.flatten()
for i, class_name in enumerate(CLASSES):
    img_files = list((GENERATED_DIR / class_name).glob("*.png"))
    if img_files:
        img = Image.open(img_files[0])
        axes[i].imshow(img)
        axes[i].set_title(class_name, fontsize=10)
        axes[i].axis("off")
plt.suptitle("Сгенерированные изображения (Stable Diffusion 1.5)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
gen_dataset = OwnImagesDataset(GENERATED_DIR, transform=eval_transform)
gen_loader = DataLoader(gen_dataset, batch_size=32, shuffle=False)

In [ ]:
gen_results = {}
for name, model in loaded_models.items():
    print(f"\n{'='*50}")
    print(f"Модель: {name} (сгенерированные данные)")
    print('='*50)
    gen_results[name] = evaluate_on_own(model, gen_loader, DEVICE, dataset_classes=CLASSES)

Качество на сгенерированных данных значительно выше, чем на ручных искажениях, значит синтетика ближе к реальному распределению. Проблемные классы: glaze, rime, sandstorm.

## Сравнение

Сравнение: свои vs аугментированные

In [ ]:
print(f"\n{'Модель':<25} | {'Свои Acc':>10} | {'Свои F1':>10} | {'Аугм Acc':>10} | {'Аугм F1':>10} | {'Ген Acc':>10} | {'Ген F1':>10}")
print("-" * 120)

for name in MODELS:
    r1 = results[name]
    r2 = aug_results[name]
    r3 = gen_results[name]
    print(f"{name:<25} | {r1['acc']:>10.4f} | {r1['f1']:>10.4f} | {r2['acc']:>10.4f} | {r2['f1']:>10.4f} |  {r3['acc']:>10.4f} | {r3['f1']:>10.4f}")

**Вывод**:
- Нет универсального лидера
- Синтетические изображения ближе к исходному распределению, чем «жёсткие» ручные искажения
- ConvNeXt Tiny — наиболее сбалансированный выбор

Grad-CAM реализация

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.fwd_hook = target_layer.register_forward_hook(self._forward_hook)
        self.bwd_hook = target_layer.register_full_backward_hook(self._backward_hook)
    
    def _forward_hook(self, module, input, output):
        self.activations = output
    
    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]
    
    def __call__(self, x, class_idx=None):
        self.model.eval()
        output = self.model(x)
        
        if class_idx is None:
            class_idx = torch.argmax(output, dim=1).item()
        
        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, class_idx] = 1
        output.backward(gradient=one_hot, retain_graph=True)
        
        pooled_gradients = torch.mean(self.gradients, dim=[0, 2, 3])
        activations = self.activations.squeeze(0)
        
        for i in range(activations.size(0)):
            activations[i, :, :] *= pooled_gradients[i]
        
        cam = torch.mean(activations, dim=0).detach().cpu()
        cam = torch.relu(cam)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        
        return cam.numpy()
    
    def remove_hooks(self):
        self.fwd_hook.remove()
        self.bwd_hook.remove()

Визуализация Grad-CAM

In [ ]:
def show_gradcam(image_path, model, model_name, device, class_names=CLASSES, true_label=None, pred_label=None, confidence=None):
    gradcam = GradCAM(model, get_target_layer(model, model_name))
    
    raw_image = Image.open(image_path).convert("RGB")
    
    preprocess = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.224])
    ])
    
    input_tensor = preprocess(raw_image).unsqueeze(0).to(device)
    
    # Если pred_label и confidence переданы — используем их, иначе пересчитываем
    if pred_label is None or confidence is None:
        with torch.no_grad():
            output = model(input_tensor)
            probs = torch.softmax(output, dim=1)[0]
            pred_class = torch.argmax(probs).item()
            confidence = probs[pred_class].item()
    else:
        pred_class = pred_label
    
    cam = gradcam(input_tensor, class_idx=pred_class)
    
    img_np = np.array(raw_image.resize((IMAGE_SIZE, IMAGE_SIZE)))
    cam_resized = cv2.resize(cam, (img_np.shape[1], img_np.shape[0]))
    
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    
    overlayed = cv2.addWeighted(img_np, 0.6, heatmap, 0.4, 0)
    
    # Строим заголовок: если есть true/pred — показываем оба
    if true_label is not None and pred_label is not None:
        title = f"Истинный: {class_names[true_label]} → Предсказание: {class_names[pred_label]}\nУверенность: {confidence:.2%}"
    else:
        title = f"Оригинал\nПредсказание: {class_names[pred_class]}\nУверенность: {confidence:.2%}"
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_np)
    axes[0].set_title(title)
    axes[0].axis('off')
    
    axes[1].imshow(cam_resized, cmap='jet')
    axes[1].set_title("Grad-CAM heatmap")
    axes[1].axis('off')
    
    axes[2].imshow(overlayed)
    axes[2].set_title("Наложение на изображение")
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    gradcam.remove_hooks()

In [ ]:
def get_target_layer(model, model_name):
    if "efficientnet_b0" in model_name:
        return model.features[-1]
    elif "convnext_tiny" in model_name:
        return model.features[-1]
    elif "resnet50" in model_name:
        return model.layer4
    else:
        raise ValueError(f"Unknown layer for {model_name}")

Анализ ошибочных предсказаний

In [ ]:
def analyze_misclassifications(model, model_name, dataset, device, class_names=CLASSES):
    loader = DataLoader(dataset, batch_size=1, shuffle=False)
    gradcam = GradCAM(model, get_target_layer(model, model_name))
    
    misclassified = []
    
    with torch.no_grad():
        for images, labels, paths in loader:
            images = images.to(device)
            output = model(images)
            pred = torch.argmax(output, dim=1).item()
            true_label = labels.item()
            
            if pred != true_label:
                misclassified.append((paths[0], true_label, pred))
    
    print(f"Найдено {len(misclassified)} ошибок из {len(dataset)} изображений")
    print(f"Error rate: {len(misclassified)/len(dataset):.2%}")
    
    for i, (path, true, pred) in enumerate(misclassified[:8]):
        print(f"\n--- Ошибка #{i+1} ---")
        print(f"Файл: {path}")
        print(f"Истинный класс: {class_names[true]}")
        print(f"Предсказание: {class_names[pred]}")
        
        show_gradcam(path, model, model_name, device, class_names, 
                    true_label=true, pred_label=pred)
    
    gradcam.remove_hooks()
    
    return misclassified

In [ ]:
print("Анализ ошибок convnext_tiny на своих данных:")
misclass_own = analyze_misclassifications(
    loaded_models["pretrained_convnext_tiny"], 
    "convnext_tiny", 
    own_dataset, 
    DEVICE
)

Анализ ошибок на аугментированных данных

In [ ]:
print("Анализ ошибок convnext_tiny на аугментированных данных:")
misclass_aug = analyze_misclassifications(
    loaded_models["pretrained_convnext_tiny"], 
    "convnext_tiny", 
    aug_dataset, 
    DEVICE
)

In [ ]:
print("\nАнализ ошибок convnext_tiny на сгенерированных данных:")
misclass_gen = analyze_misclassifications(
    loaded_models["pretrained_convnext_tiny"],
    "convnext_tiny",
    gen_dataset,
    DEVICE
)

Матрицы ошибок (confusion matrices)

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(title, fontsize=14)
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45, ha='right')
    plt.yticks(tick_marks, class_names)
    
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()

In [ ]:
for name in MODELS:
    plot_confusion_matrix(
        results[name]["y_true"],
        results[name]["y_pred"],
        CLASSES,
        f"Confusion Matrix — {name} (свои данные)"
    )

In [ ]:
for name in MODELS:
    plot_confusion_matrix(
        aug_results[name]["y_true"],
        aug_results[name]["y_pred"],
        CLASSES,
        f"Confusion Matrix — {name} (аугментированные данные)"
    )

In [ ]:
for name in MODELS:
    plot_confusion_matrix(
        gen_results[name]["y_true"],
        gen_results[name]["y_pred"],
        CLASSES,
        f"Confusion Matrix — {name} (сгенерированные данные)"
    )

Критические классы для улучшения: frost, glaze, rime, rain.

Анализ результатов и предложения по дообучению

In [ ]:
def analyze_common_errors(misclassified, class_names=CLASSES):
    error_pairs = []
    for path, true_label, pred_label in misclassified:
        error_pairs.append((class_names[true_label], class_names[pred_label]))
    
    from collections import Counter
    error_counts = Counter(error_pairs)
    
    print("Самые частые ошибки (истинный -> предсказанный):")
    for (true_cls, pred_cls), count in error_counts.most_common(10):
        print(f"  {true_cls:>12} -> {pred_cls:>12}: {count} раз")

In [ ]:
print("=== Частые ошибки на своих данных ===")
analyze_common_errors(misclass_own)

In [ ]:
print("\n=== Частые ошибки на аугментированных данных ===")
analyze_common_errors(misclass_aug)

In [ ]:
print("\n=== Частые ошибки на сгенерированных данных ===")
analyze_common_errors(misclass_gen)

Способы дообучения для улучшения результатов:
1. Целевые аугментации для проблемных групп
2. Hard Negative Mining. Собрать в отдельные батчи пары изображений, которые модель чаще всего путает (например, frost vs glaze, rain vs dew). Временно увеличить вес таких примеров при обучении или использовать контрастивную функцию потерь — это заставит модель учиться на «сложных» случаях целенаправленно.
3. Присвоить больший вес классам с низким F1
4. Постепенная разморозка слоёв
5. Расширение датасета
6. При инференсе делать несколько предсказаний для одного изображения с небольшими аугментациями (повороты, изменение яркости) и усреднять результаты — это даёт бесплатный прирост точности на 2–5%
7. Можно добавить простое правило: при низкой уверенности модели проверять наличие контекстных признаков (температура, время суток, гео-метаданные) и корректировать предсказание.